In [ ]:
from pydantic import BaseModel
import os 
import json
import numpy as np
import polars as pl
from movedb.core import Trial
from movedb.file_io import sto_to_df
from src.grouping import trim_and_spline
root_dir = os.path.join('C:\\', 'Users', 'hpb7kr', 'OneDrive - University of Virginia', 'Shared Documents - MAMP Lab Folder', 'General', 'MoCapData', 'ViconData', 'Rats')

# Process all control group results
print("Processing control group results...")
control_results_file = os.path.join(root_dir, 'control_group_results.json')

with open(control_results_file, 'r') as f:
    control_results = json.load(f)

spline_points = 100
gait_percentage = pl.Series('gait_percentage', np.linspace(0, 100, spline_points))

control_ik = {"Left": [], "Right": []}
control_id = {"Left": [], "Right": []}
control_grf = {"Left": [], "Right": []}

left_stance_swing = [
    ('Left', 'Foot Strike'),
    ('Right', 'Foot Off'),
    ('Right', 'Foot Strike'),
    ('Left', 'Foot Off'),
    ('Left', 'Foot Strike'),
    ('Right', 'Foot Off'),
    ('Right', 'Foot Strike'),
]

right_stance_swing = [
    ('Right', 'Foot Strike'),
    ('Left', 'Foot Off'),
    ('Left', 'Foot Strike'),
    ('Right', 'Foot Off'),
    ('Right', 'Foot Strike'),
    ('Left', 'Foot Off'),
    ('Left', 'Foot Strike'),
]

for classification, subjects in control_results.items():
    for subject, sessions in subjects.items():
        for session, session_data in sessions.items():
            if session_data is None or not isinstance(session_data, dict):
                continue
            walking_trials = session_data.get('walking_trials', [])
            session_ik: dict[str, list[pl.DataFrame]]= {"Left": [], "Right": []}
            session_id: dict[str, list[pl.DataFrame]] = {"Left": [], "Right": []}
            session_grf: dict[str, list[pl.DataFrame]] = {"Left": [], "Right": []}
            for trial_pkl, files in walking_trials:
                ik_file = files.get('ik', None)
                id_file = files.get('id', None)
                grf_file = files.get('grf', None)
                trial = Trial.from_pkl(trial_pkl)
                mass = trial.parameters.get('Mass', 0.1)
                # Spline to stance swing
                for side in ['Left', 'Right']:
                    phases = trial.get_event_sequences(right_stance_swing if side == 'Right' else left_stance_swing)
                    for phase in phases:
                        stance_phase = phase[:2]
                        swing_phase = phase[1:]
                        stance_times = [event.get_time(trial.points.rate) for event in stance_phase]
                        swing_times = [event.get_time(trial.points.rate) for event in swing_phase]
                        if ik_file:
                            ik_df, _ = sto_to_df(ik_file)
                            stance_ik = trim_and_spline(ik_df, start=stance_times[0], end=stance_times[-1])
                            swing_ik = trim_and_spline(ik_df, start=swing_times[0], end=swing_times[-1])
                            spline_ik = pl.concat([stance_ik, swing_ik]).sort('time')
                            spline_ik = spline_ik.with_columns(gait_percentage)
                            session_ik[side].append(spline_ik)
                        if id_file:
                            id_df, _ = sto_to_df(id_file)
                            stance_id = trim_and_spline(id_df, start=stance_times[0], end=stance_times[-1])
                            swing_id = trim_and_spline(id_df, start=swing_times[0], end=swing_times[-1])
                            spline_id = pl.concat([stance_id, swing_id]).sort('time')
                            spline_id = spline_id.with_columns(pl.exclude('time') / mass)
                            spline_id = spline_id.with_columns(gait_percentage)
                            session_id[side].append(spline_id)
                        if grf_file:
                            grf_df, _ = sto_to_df(grf_file)
                            stance_grf = trim_and_spline(grf_df, start=stance_times[0], end=stance_times[-1])
                            swing_grf = trim_and_spline(grf_df, start=swing_times[0], end=swing_times[-1])
                            spline_grf = pl.concat([stance_grf, swing_grf]).sort('time')
                            # Only divide columns starting with 'force<int>_v' or 'moment<int>_' by mass
                            force_cols = [col for col in spline_grf.columns if pl.col(col).meta.root_names()[0].startswith('force') and '_v' in col]
                            moment_cols = [col for col in spline_grf.columns if pl.col(col).meta.root_names()[0].startswith('moment')]
                            cols_to_divide = force_cols + moment_cols
                            if cols_to_divide:
                                spline_grf = spline_grf.with_columns([
                                    pl.col(col) / mass for col in cols_to_divide
                                ])
                            # add gait percentage column
                            spline_grf = spline_grf.with_columns(gait_percentage)
                            session_grf[side].append(spline_grf)
            # Average the session data across trials
            for side in ['Left', 'Right']:
                if session_ik[side]:
                    control_ik[side].append(pl.concat(session_ik[side]).group_by('time').agg(pl.mean()))
                if session_id[side]:
                    control_id[side].append(pl.concat(session_id[side]).group_by('time').agg(pl.mean()))
                if session_grf[side]:
                    control_grf[side].append(pl.concat(session_grf[side]).group_by('time').agg(pl.mean()))
            

Processing control group results...


AttributeError: 'NoneType' object has no attribute 'get'